In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

import joblib

In [2]:
df = pd.read_csv("../outputs/cleaned_data.csv")

df.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


In [4]:
X = df.drop(columns=["Diagnosis", "PatientID", "DoctorInCharge"])

y = df["Diagnosis"]

print(X.shape)
print(y.shape)

(2149, 32)
(2149,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(1719, 32)
(430, 32)


In [8]:
tree = DecisionTreeClassifier(random_state=42)

tree.fit(X_train_scaled, y_train)

train_acc = accuracy_score(
    y_train,
    tree.predict(X_train_scaled)
)

test_acc = accuracy_score(
    y_test,
    tree.predict(X_test_scaled)
)

print("Training Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)

Training Accuracy: 1.0
Testing Accuracy : 0.9


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

controlled_tree.fit(X_train_scaled, y_train)

train_pred = controlled_tree.predict(X_train_scaled)
test_pred = controlled_tree.predict(X_test_scaled)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("Controlled Decision Tree")
print("Training Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)

Controlled Decision Tree
Training Accuracy: 0.9610238510762071
Testing Accuracy : 0.9302325581395349


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Gini Tree
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train)

gini_pred = gini_tree.predict(X_test_scaled)

gini_acc = accuracy_score(y_test, gini_pred)

# Entropy Tree
entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_train)

entropy_pred = entropy_tree.predict(X_test_scaled)

entropy_acc = accuracy_score(y_test, entropy_pred)

print("Gini Accuracy :", gini_acc)
print("Entropy Accuracy :", entropy_acc)

Gini Accuracy : 0.9232558139534883
Entropy Accuracy : 0.9465116279069767


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

train_pred = rf_model.predict(X_train_scaled)
test_pred = rf_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, y_prob)

print("Training Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)
print("ROC-AUC:", auc)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(5)

Training Accuracy: 0.987783595113438
Testing Accuracy : 0.9488372093023256
ROC-AUC: 0.9401741764483151


,Feature,Importance
23,FunctionalAssessment,0.194332
26,ADL,0.165438
22,MMSE,0.136758
24,MemoryComplaints,0.095701
25,BehavioralProblems,0.047430


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

controlled_tree.fit(X_train_scaled, y_train)

train_pred = controlled_tree.predict(X_train_scaled)
test_pred = controlled_tree.predict(X_test_scaled)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("Controlled Decision Tree")
print("Training Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)

Controlled Decision Tree
Training Accuracy: 0.9610238510762071
Testing Accuracy : 0.9302325581395349


In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)

train_pred = gb_model.predict(X_train_scaled)
test_pred = gb_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

y_prob = gb_model.predict_proba(X_test_scaled)[:,1]
auc = roc_auc_score(y_test, y_prob)

print("Gradient Boosting")
print("Training Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)
print("ROC-AUC:", auc)

Gradient Boosting
Training Accuracy: 0.9668411867364747
Testing Accuracy : 0.9465116279069767
ROC-AUC: 0.9463389814464217


In [13]:
lowest_features = importance.sort_values(
    by="Importance"
).head(5)

print(lowest_features)

               Feature  Importance
14          HeadInjury    0.002159
29  PersonalityChanges    0.002445
12            Diabetes    0.002653
13          Depression    0.002691
27           Confusion    0.002949


In [14]:
# Remove the 5 least important features

low_features = [
    "HeadInjury",
    "PersonalityChanges",
    "Diabetes",
    "Depression",
    "Confusion"
]

X_reduced = X.drop(columns=low_features)

X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_red = scaler.fit_transform(X_train_red)
X_test_red = scaler.transform(X_test_red)

rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_red, y_train_red)

prob_reduced = rf_reduced.predict_proba(X_test_red)[:, 1]

auc_reduced = roc_auc_score(y_test_red, prob_reduced)

print("Full Model AUC :", auc)
print("Reduced Model AUC :", auc_reduced)

Full Model AUC : 0.9463389814464217
Reduced Model AUC : 0.9419017417644833


In [15]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc"
    )

    results.append([
        name,
        scores.mean(),
        scores.std()
    ])

cv_results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Mean AUC",
        "Std AUC"
    ]
)

cv_results

c:\Users\Rethi\OneDrive\Desktop\Neurosage\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Rethi\OneDrive\Desktop\Neurosage\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.or

,Model,Mean AUC,Std AUC
0,Logistic Regression,0.894247,0.011598
1,Decision Tree,0.936189,0.018929
2,Random Forest,0.952997,0.006567
3,Gradient Boosting,0.950633,0.009823


In [16]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest CV AUC:")
print(grid.best_score_)

Best Parameters:
{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best CV AUC:
0.9561810422785275


In [17]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

learning_results = []

best_pipeline = grid.best_estimator_

for f in fractions:

    n = int(f * len(X_train))

    X_sub = X_train.iloc[:n]
    y_sub = y_train.iloc[:n]

    best_pipeline.fit(X_sub, y_sub)

    train_prob = best_pipeline.predict_proba(X_sub)[:, 1]
    test_prob = best_pipeline.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_sub, train_prob)
    test_auc = roc_auc_score(y_test, test_prob)

    learning_results.append([f, train_auc, test_auc])

learning_curve = pd.DataFrame(
    learning_results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]
)

learning_curve

,Training Fraction,Training AUC,Test AUC
0,0.2,1.0,0.940044
1,0.4,1.0,0.942493
2,0.6,1.0,0.939464
3,0.8,1.0,0.942789
4,1.0,1.0,0.940233


In [18]:
import joblib

best_pipeline = grid.best_estimator_

joblib.dump(best_pipeline, "best_model.pkl")

print("best_model.pkl saved successfully!")

best_model.pkl saved successfully!


In [19]:
loaded_model = joblib.load("best_model.pkl")

sample = X.iloc[:2]

prediction = loaded_model.predict(sample)

print("Predictions:")
print(prediction)

Predictions:
[0 0]
